In [1]:
import pandas as pd

df_cp = pd.read_csv("../data/processed/race_driver_labels.csv")
df_cp.head()

,raceId,driverId,constructorId,grid,positionOrder,points,statusId,year,round,circuitId,...,avgPitDuration_ms,driverName_x,constructorName_x,driverName_y,constructorName_y,driverName,constructorName,finishPosition,avgLapTime_s,constructorPoints
0,18,1,1,1,1,10.0,1,2008,1,1,...,0.0,Lewis Hamilton,McLaren,Lewis Hamilton,McLaren,Lewis Hamilton,McLaren,1,98.114069,14.0
1,18,2,2,5,2,8.0,1,2008,1,1,...,0.0,Nick Heidfeld,BMW Sauber,Nick Heidfeld,BMW Sauber,Nick Heidfeld,BMW Sauber,2,98.208517,8.0
2,18,3,3,7,3,6.0,1,2008,1,1,...,0.0,Nico Rosberg,Williams,Nico Rosberg,Williams,Nico Rosberg,Williams,3,98.254810,9.0
3,18,4,4,11,4,5.0,1,2008,1,1,...,0.0,Fernando Alonso,Renault,Fernando Alonso,Renault,Fernando Alonso,Renault,4,98.410293,5.0
4,18,5,1,3,5,4.0,1,2008,1,1,...,0.0,Heikki Kovalainen,McLaren,Heikki Kovalainen,McLaren,Heikki Kovalainen,McLaren,5,98.424655,14.0


In [2]:
df_cp["constructorPoints"].describe()

count    26759.000000
mean         4.464607
std          7.797860
min          0.000000
25%          0.000000
50%          0.000000
75%          6.000000
max         66.000000
Name: constructorPoints, dtype: float64

In [3]:
features_cp = [
    "qualifyingPosition",   # starting advantage
    "finishPosition",       # proxy for race outcome strength
    "pitStopCount",         # race strategy
    "grid"                  # starting grid position
]

In [4]:
df_cp[features_cp + ["constructorPoints"]].isna().sum()

qualifyingPosition    16265
finishPosition            0
pitStopCount              0
grid                      0
constructorPoints         0
dtype: int64

In [5]:
df_cp_model = df_cp.dropna(
    subset=features_cp + ["constructorPoints"]
).copy()

print("Rows before:", len(df_cp))
print("Rows after:", len(df_cp_model))

Rows before: 26759
Rows after: 10494


In [6]:
X_cp = df_cp_model[features_cp]
y_cp = df_cp_model["constructorPoints"]

In [7]:
from sklearn.model_selection import train_test_split

X_train_cp, X_test_cp, y_train_cp, y_test_cp = train_test_split(
    X_cp, y_cp, test_size=0.2, random_state=42
)

In [8]:
from sklearn.linear_model import LinearRegression

lr_cp = LinearRegression()
lr_cp.fit(X_train_cp, y_train_cp)

LinearRegression()

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_cp_lr = lr_cp.predict(X_test_cp)

mae_cp_lr = mean_absolute_error(y_test_cp, y_pred_cp_lr)
rmse_cp_lr = np.sqrt(mean_squared_error(y_test_cp, y_pred_cp_lr))
r2_cp_lr = r2_score(y_test_cp, y_pred_cp_lr)

print("Constructor Points — Linear Regression")
print(f"MAE: {mae_cp_lr:.2f}")
print(f"RMSE: {rmse_cp_lr:.2f}")
print(f"R²: {r2_cp_lr:.3f}")

Constructor Points — Linear Regression
MAE: 5.48
RMSE: 7.30
R²: 0.496


In [10]:
from sklearn.ensemble import RandomForestRegressor

rf_cp = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_cp.fit(X_train_cp, y_train_cp)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [11]:
y_pred_cp_rf = rf_cp.predict(X_test_cp)

mae_cp_rf = mean_absolute_error(y_test_cp, y_pred_cp_rf)
rmse_cp_rf = np.sqrt(mean_squared_error(y_test_cp, y_pred_cp_rf))
r2_cp_rf = r2_score(y_test_cp, y_pred_cp_rf)

print("Constructor Points — Random Forest")
print(f"MAE: {mae_cp_rf:.2f}")
print(f"RMSE: {rmse_cp_rf:.2f}")
print(f"R²: {r2_cp_rf:.3f}")

Constructor Points — Random Forest
MAE: 3.46
RMSE: 5.54
R²: 0.710


Random Forest outperforms Linear Regression
Constructor points are influenced by non-linear interactions
Strategy + starting position + finish outcome matter